In [ ]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)


# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

**1. Cats and Dogs Dataset Downloading**

In [ ]:
# Download the Dataset and create a directory PetImages with two folders: Cat and Dog
# https://www.kaggle.com/datasets/tongpython/cat-and-dog
# https://www.microsoft.com/en-us/download/details.aspx?id=54765
# Each of the folders contains images from one class

!curl -O https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

!unzip -q kagglecatsanddogs_5340.zip

!rm kagglecatsanddogs_5340.zip


In [ ]:
# Some images are corrupted, so they have to be removed

import os

num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join("PetImages", folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, "rb")
            is_jfif = b"JFIF" in fobj.peek(10)
        finally:
            fobj.close()

        if not is_jfif:
            num_skipped += 1
            # Delete corrupted image
            os.remove(fpath)

print(f"Deleted {num_skipped} images.")

In [ ]:

cat_dir = 'PetImages/Cat'
dog_dir = 'PetImages/Dog'

num_cats = len(os.listdir(cat_dir))
num_dogs = len(os.listdir(dog_dir))

print('Cat images:', num_cats)
print('Dog images: ', num_dogs)

**2. Dataset Preprocessing**

In [ ]:
# Create datasets for training and testing

image_size = (128, 128)
batch_size = 32

train_ds, test_ds = keras.utils.image_dataset_from_directory(
    "PetImages",
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
)

class_names = train_ds.class_names
print(class_names)

In [ ]:
# Visualize a few examples

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy()/255.0) # the image from the dataset is transformed into a numpy array
        plt.title(class_names[labels[i]])
        plt.axis("off")

**3. Creating and Training a Convolutional Neural Network**

In [ ]:
# Create a Convolutional NN with name my_first_CNN
# It relies on the Keras Sequential API: https://keras.io/api/models/
# When creating a NN, it adds layeys one by one, using the layer class: https://keras.io/api/layers/base_layer/

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

my_first_CNN = keras.models.Sequential([
    layers.InputLayer(shape=(128,128,3)),
    layers.Rescaling(1./255),
    layers.Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu',padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu',padding='same'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation="sigmoid")
])




In [ ]:
# Present a summary of the network architecture
# Confirm the architecture and the number of parameters

my_first_CNN.summary()

In [ ]:

# A NN has to be compiled by calling the compile() method: https://keras.io/api/models/model_training_apis/
# Three components have to be defined (recall how backpropagation is appplied during training):
# 1. the Optimizer to be used in training
# 2. The loss function
# 3. The evaluation metric

my_first_CNN.compile(loss='binary_crossentropy',
              optimizer=keras.optimizers.SGD(),
              metrics=['accuracy'])


# Training is performed by calling the fit() method: https://keras.io/api/models/model_training_apis/

history = my_first_CNN.fit(train_ds, epochs=20)


In [ ]:
# Evolution of Results

import pandas as pd

x = pd.DataFrame(history.history,columns = ['accuracy'])
x.plot(figsize=(8, 5))
plt.ylim(0.5, 1)
plt.grid(True)
plt.show()

**4. CNN Evaluation**

In [ ]:
# Evaluate the neural network on the test dataset

my_first_CNN.evaluate(test_ds)

**5. Try with your own images**

1. Upload a jpg image to the working directory
2. Specify the name of the image
3. Apply the CNN and confirm the prediction

In [ ]:
Image_Name = 'A.jpeg' #@param {type:"string"}

img = keras.utils.load_img(Image_Name, target_size=image_size)
plt.imshow(img)

img_array = keras.utils.img_to_array(img)
img_array = keras.ops.expand_dims(img_array, 0)  # Create batch axis

predictions = my_first_CNN.predict(img_array)
score = float(keras.ops.sigmoid(predictions[0][0]))
print(f"This image is {100 * (1 - score):.2f}% cat and {100 * score:.2f}% dog.")

**6. Fighting Overfitting**

In [ ]:
# Create an enhanced Convolutional NN with name my_second_CNN
# It relies on the Keras Sequential API: https://keras.io/api/models/
# When creating a NN, it adds layeys one by one, using the layer class: https://keras.io/api/layers/base_layer/
# It has Dropout Data Augmentation layers

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

my_second_CNN = keras.models.Sequential([
    layers.InputLayer(shape=(128,128,3)),
    layers.Rescaling(1./255),
    layers.RandomRotation(factor=0.1),
    layers.Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu',padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu',padding='same'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.25),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.25),
    layers.Dense(1, activation="sigmoid")
])

In [ ]:
# Compile and Train

my_second_CNN.compile(loss='binary_crossentropy',
              optimizer=keras.optimizers.SGD(),
              metrics=['accuracy'])


history = my_second_CNN.fit(train_ds, epochs=20)

In [ ]:
# Evolution of Results

import pandas as pd

x = pd.DataFrame(history.history,columns = ['accuracy'])
x.plot(figsize=(8, 5))
plt.ylim(0.5, 1)
plt.grid(True)
plt.show()

In [ ]:
# Evaluate the neural network on the test dataset

my_second_CNN.evaluate(test_ds)
